# Appendix 1. 데이터 분석을 위한 Python 기초

## 1. Goal

이 notebook은 pandas와 NumPy 코드를 읽기 전에 필요한 Python 문법과 객체 동작을 익히는 자료입니다. 기본 문법을 살핀 뒤 Python Data Model을 통해 객체의 identity·type·value, name binding, mutable과 immutable을 구분합니다. 이어서 주요 container, slicing과 반복, 함수, 예외를 작은 합성 ICU 예제로 확인합니다.

모든 Python 기능을 훑는 것이 아니라 이후 appendix와 EDA notebook에서 반복해서 사용하는 표현을 스스로 읽고 수정할 수 있는 수준을 목표로 합니다.

## 2. Setup

프로젝트는 Python 3.11을 사용합니다.

표준 라이브러리만 사용하며 외부 파일은 읽지 않습니다. 각 cell에서 값만 보지 말고 `type`, `id`, 객체 공유 여부와 반환 객체도 함께 확인하세요.

In [ ]:
import copy
import sys

sys.version.split()[0]


## 3. Steps

각 `###` 절은 Python의 관련 개념군을, `####` 절은 하나의 실행 예제로 확인할 세부 학습 단위를 나타냅니다.

### 3-1. 값, 객체와 이름

#### 3-1-1. 기본 타입과 None

Python 변수는 값을 담는 고정 타입 상자가 아니라 객체를 가리키는 이름입니다. `int`, `float`, `bool`, `str`은 자주 쓰는 기본 타입이고, `None`은 값이 아직 없거나 해당하지 않음을 명시합니다. `None` 여부는 `==`보다 `is None`으로 확인합니다.

In [ ]:
patient_id = "P101"
measurement_count = 3
heart_rate = 84.5
is_complete = False
missing_value = None

type_summary = {
    "patient_id": type(patient_id).__name__,
    "measurement_count": type(measurement_count).__name__,
    "heart_rate": type(heart_rate).__name__,
    "is_complete": type(is_complete).__name__,
    "missing_value": type(missing_value).__name__,
}
print(type_summary)
print({"has_missing_value": missing_value is None})


### 3-2. Python Data Model

Python Data Model은 Python 코드에 등장하는 값이 어떤 객체로 표현되고, 연산이 그 객체와 어떻게 상호작용하는지 설명합니다. 이 구조를 알면 pandas 객체를 다른 변수에 대입하거나 함수에 넘긴 뒤 원본이 함께 바뀌는 이유를 판단하기 쉬워집니다.

#### 3-2-1. identity, type, value와 name binding

모든 객체에는 생성 후 바뀌지 않는 identity, 지원하는 연산을 정하는 type, 그리고 현재 value가 있습니다. `id()`는 identity를 나타내는 정수를, `type()`은 type 객체를 반환합니다. `==`는 value가 같은지 비교하고 `is`는 두 이름이 같은 객체를 가리키는지 비교합니다. 대입문은 객체를 자동으로 복사하지 않고 name을 객체에 bind합니다.

In [ ]:
heart_rate_value = 84
integer_identity_before = id(heart_rate_value)
heart_rate_value += 1
integer_identity_after = id(heart_rate_value)

original_values = [78, 82, 86]
alias_values = original_values
copied_values = original_values.copy()

list_identity_before = id(original_values)
alias_values.append(90)
list_identity_after = id(original_values)
identity_summary = {
    "integer_rebinding_created_another_object": (
        integer_identity_before != integer_identity_after
    ),
    "list_mutation_kept_identity": (
        list_identity_before == list_identity_after
    ),
    "alias_is_original": alias_values is original_values,
    "copy_is_original": copied_values is original_values,
    "copy_has_old_value": copied_values == [78, 82, 86],
    "original": original_values,
    "copy": copied_values,
}
identity_summary


#### 3-2-2. mutable과 immutable

mutable 객체는 생성된 뒤에도 같은 identity를 유지한 채 내용을 바꿀 수 있습니다. list, dict, set이 대표적입니다. 반면 int, float, bool, str, tuple, frozenset, bytes 같은 immutable 객체는 생성 후 객체 자체의 내용을 바꿀 수 없습니다. `text += ...`처럼 변경해 보이는 코드도 실제로는 새 객체를 만든 뒤 name을 다시 bind합니다.

tuple이 immutable이라는 말은 tuple이 직접 참조하는 원소 구성을 바꿀 수 없다는 뜻입니다. tuple 안에 mutable 객체가 들어 있다면 그 객체의 내용은 바뀔 수 있습니다. 이 구분은 중첩된 데이터를 다룰 때 특히 중요합니다.

In [ ]:
unit_name = "ICU"
string_identity_before = id(unit_name)
unit_name += "-A"
string_identity_after = id(unit_name)

mutable_measurements = [78, 82]
list_identity_before_append = id(mutable_measurements)
mutable_measurements.append(86)
list_identity_after_append = id(mutable_measurements)

nested_tuple = ("P101", [78, 82])
nested_tuple[1].append(86)

mutability_summary = {
    "string_rebinding_created_another_object": (
        string_identity_before != string_identity_after
    ),
    "list_mutation_kept_identity": (
        list_identity_before_append == list_identity_after_append
    ),
    "mutable_item_inside_tuple": nested_tuple,
}
mutability_summary


#### 3-2-3. shallow copy와 deep copy

shallow copy는 바깥 container만 새로 만들고 안쪽 객체는 원본과 공유합니다. 따라서 중첩된 list나 dict를 수정하면 원본에도 변경이 보일 수 있습니다. `copy.deepcopy()`는 안쪽 객체까지 재귀적으로 복사합니다. 항상 deep copy를 쓰기보다는, 어느 수준까지 독립되어야 하는지 먼저 정해야 불필요한 메모리 사용과 의도하지 않은 공유를 함께 피할 수 있습니다.

In [ ]:
patient_record = {"P101": {"values": [78, 82]}}
shallow_record = patient_record.copy()
deep_record = copy.deepcopy(patient_record)

shallow_record["P101"]["values"].append(86)
copy_summary = {
    "outer_dict_is_shared": shallow_record is patient_record,
    "inner_dict_is_shared": (
        shallow_record["P101"] is patient_record["P101"]
    ),
    "original_after_shallow_mutation": patient_record,
    "deep_copy_stayed_independent": deep_record,
}
copy_summary


#### 3-2-4. hashability와 dict·set key

hashable 객체는 lifetime 동안 hash 값이 바뀌지 않고, 서로 같다고 비교되는 객체끼리 같은 hash 값을 가집니다. dict key와 set 원소에는 hashable 객체만 사용할 수 있습니다. 문자열이나 숫자, hashable 원소로만 구성된 tuple은 보통 사용할 수 있지만 list와 dict는 사용할 수 없습니다. immutable이라고 해서 언제나 hashable한 것은 아닙니다. 예를 들어 list를 담은 tuple은 그 list 때문에 hashable하지 않습니다.

In [ ]:
measurement_key = ("P101", "HR")
measurements_by_key = {measurement_key: 84.5}
measurement_key_hash = hash(measurement_key)

try:
    hash(("P101", [78, 82]))
except TypeError as error:
    unhashable_error = type(error).__name__

print(
    {
        "lookup": measurements_by_key[("P101", "HR")],
        "hash": measurement_key_hash,
        "tuple_with_list": unhashable_error,
    }
)


#### 3-2-5. 함수 인자와 mutable 기본값

함수를 호출하면 argument가 가리키는 객체가 parameter name에 bind됩니다. 함수 안에서 mutable 객체를 직접 바꾸면 호출한 쪽에서도 변경을 볼 수 있습니다. 또 기본값 객체는 함수를 정의할 때 한 번 만들어지므로 `bucket=[]` 같은 mutable 기본값은 여러 호출 사이에 상태를 공유합니다. 호출마다 새 container가 필요하면 기본값으로 `None`을 두고 함수 안에서 생성합니다.

In [ ]:
def append_observation(values: list[float], value: float) -> None:
    """Append one observation to the caller-owned list."""
    values.append(value)


def unsafe_collect(value: float, bucket: list[float] = []) -> list[float]:
    """Demonstrate state retained by a mutable default argument."""
    bucket.append(value)
    return bucket


def collect_observation(
    value: float,
    bucket: list[float] | None = None,
) -> list[float]:
    """Return observations without sharing a default list across calls."""
    result = [] if bucket is None else bucket
    result.append(value)
    return result


caller_values = [78.0, 82.0]
append_observation(caller_values, 86.0)
unsafe_first = unsafe_collect(78.0)
unsafe_second = unsafe_collect(82.0)
safe_first = collect_observation(78.0)
safe_second = collect_observation(82.0)

print(
    {
        "caller_values": caller_values,
        "unsafe_calls_share_list": unsafe_first is unsafe_second,
        "unsafe_default": unsafe_collect.__defaults__,
        "safe_calls_share_list": safe_first is safe_second,
    }
)


#### 3-2-6. protocol과 special method

`len(values)`, `value in values`, 반복, indexing은 서로 달라 보이지만 객체의 type이 제공하는 protocol을 사용합니다. 예를 들어 `len()`은 `__len__`, membership 검사는 주로 `__contains__`, indexing은 `__getitem__`과 연결됩니다. 이런 `__이름__` 형태를 special method라고 합니다. 일반 코드에서는 special method를 직접 부르기보다 `len()`, `in`, `for`, `[]`처럼 의도가 드러나는 문법과 built-in function을 사용합니다.

In [ ]:
protocol_values = [78, 82, 86]
protocol_summary = {
    "length": len(protocol_values),
    "length_via_special_method": type(protocol_values).__len__(
        protocol_values
    ),
    "contains_82": 82 in protocol_values,
    "first_item": protocol_values[0],
    "iterated_items": list(iter(protocol_values)),
}
protocol_summary


### 3-3. Container 자료형

#### 3-3-1. list와 tuple

list는 순서가 있고 수정 가능한 값 모음입니다. tuple도 순서가 있지만 생성 후 원소 구성을 바꾸지 않습니다. 측정값처럼 변하는 모음에는 list를, 좌표나 고정 schema처럼 한 덩어리로 전달할 값에는 tuple을 우선 고려합니다.

In [ ]:
patient_ids = ["P101", "P102", "P103"]
measurement = ("P101", "HR", 84.5)

patient_ids.append("P104")
measurement_patient, measurement_parameter, measurement_value = measurement
print(
    {
        "patients": patient_ids,
        "measurement_type": type(measurement).__name__,
        "unpacked": (
            measurement_patient,
            measurement_parameter,
            measurement_value,
        ),
    }
)


#### 3-3-2. dict와 set

dict는 unique key를 값에 연결하고, set은 중복 없는 원소를 표현합니다. `dict.get`은 key가 없을 때 기본값을 줄 수 있고, set의 교집합·차집합은 실제 관측 항목과 기대 항목을 비교할 때 유용합니다.

In [ ]:
patient_summary = {
    "patient_id": "P101",
    "measurement_count": 4,
    "missing_count": 1,
}
expected_parameters = {"HR", "Lactate", "FiO2"}
observed_parameters = {"HR", "Lactate", "HR"}

container_summary = {
    "unknown_value": patient_summary.get("target", "unknown"),
    "covered": sorted(expected_parameters & observed_parameters),
    "missing": sorted(expected_parameters - observed_parameters),
}
container_summary


### 3-4. 선택과 반복

#### 3-4-1. Indexing, slicing과 unpacking

Python sequence는 0부터 시작하는 위치 index를 사용합니다. slice의 끝 위치는 포함하지 않으며, 음수 index는 뒤에서부터 셉니다. unpacking의 `*`는 남은 원소를 모아 받습니다.

In [ ]:
ordered_values = [72, 78, 84, 90, 96]
first_value = ordered_values[0]
last_value = ordered_values[-1]
middle_values = ordered_values[1:4]
first, *middle, last = ordered_values

print(
    {
        "first": first_value,
        "last": last_value,
        "slice": middle_values,
        "unpacked_middle": middle,
    }
)


#### 3-4-2. for, enumerate, zip과 comprehension

`enumerate`는 위치와 값을 함께, `zip`은 여러 iterable의 같은 위치 값을 묶어 순회합니다. comprehension은 변환과 필터가 짧고 명확할 때 사용합니다. 조건이 복잡하거나 side effect가 있으면 일반 for 문이 더 읽기 쉽습니다.

In [ ]:
patients = ["P101", "P102", "P103", "P104"]
values = [78.0, None, 86.0, 90.0]

numbered_patients = [
    {"position": position, "patient_id": patient}
    for position, patient in enumerate(patients, start=1)
]
observed_by_patient = {
    patient: value
    for patient, value in zip(patients, values, strict=True)
    if value is not None
}
print(numbered_patients)
observed_by_patient


### 3-5. 조건과 함수

#### 3-5-1. 조건식, any와 all

빈 container는 false로 평가됩니다. `any`는 하나라도 참인지, `all`은 모두 참인지 확인합니다. 데이터 검증에서는 각 조건이 무엇을 뜻하는지 이름을 붙여 표현하면 복합 조건을 읽기 쉬워집니다.

In [ ]:
required_columns = {"patient_id", "parameter", "value"}
actual_columns = {"patient_id", "parameter", "value", "minute"}
values_to_check = [78.0, 82.0, None]

has_required_columns = required_columns <= actual_columns
has_missing = any(value is None for value in values_to_check)
all_observed_are_positive = all(
    value > 0 for value in values_to_check if value is not None
)
print(
    {
        "has_required_columns": has_required_columns,
        "has_missing": has_missing,
        "all_observed_are_positive": all_observed_are_positive,
    }
)


#### 3-5-2. 함수, 반환값과 keyword-only 인자

함수는 한 가지 계산 규칙에 이름을 붙입니다. type hint는 실행 시 강제되는 검증이 아니라 독자와 도구를 위한 설명입니다. `*` 뒤 인자는 이름을 명시해야 하므로 의미가 다른 숫자 인자의 순서 실수를 줄입니다.

In [ ]:
def summarize_values(
    values: list[float],
    *,
    precision: int = 1,
) -> dict[str, float]:
    """Return bounded summary statistics for non-empty observed values."""
    if not values:
        raise ValueError("values must not be empty")
    return {
        "minimum": round(min(values), precision),
        "maximum": round(max(values), precision),
        "mean": round(sum(values) / len(values), precision),
    }


value_summary = summarize_values([78.0, 82.0, 86.0], precision=2)
value_summary


#### 3-5-3. lambda와 key 함수

함수도 객체이므로 다른 함수의 인자로 전달할 수 있습니다. 짧은 일회성 변환은 `lambda`로 표현할 수 있으며, `sorted(..., key=...)`처럼 정렬 기준을 넘길 때 자주 사용합니다. 여러 단계이거나 재사용할 로직은 이름 있는 함수로 작성합니다.

In [ ]:
feature_rows = [
    {"feature": "HR", "missing_rate": 0.05},
    {"feature": "Lactate", "missing_rate": 0.25},
    {"feature": "FiO2", "missing_rate": 0.10},
]
ranked_features = sorted(
    feature_rows,
    key=lambda row: row["missing_rate"],
    reverse=True,
)
ranked_features


### 3-6. 오류와 검증

#### 3-6-1. 예외를 읽고 필요한 오류만 처리하기

예외는 실패 종류와 발생 위치를 전달합니다. 예상하고 복구할 수 있는 구체적인 예외만 처리하고, 모든 오류를 `except Exception`으로 숨기지 않습니다. 아래에서는 정수 변환 실패를 명시적인 상태로 바꿉니다.

In [ ]:
raw_target = "not-a-number"

try:
    parsed_target = int(raw_target)
except ValueError as error:
    parsed_target = None
    parsing_error = type(error).__name__

print({"parsed_target": parsed_target, "error": parsing_error})


#### 3-6-2. assert로 학습 결과 확인하기

`assert`는 개발 중 반드시 참이어야 하는 가정을 빠르게 확인합니다. 외부 입력의 사용자 친화적 검증을 대신하지는 않지만, notebook 예제의 shape·개수·반환 타입을 확인하는 데 적합합니다.

In [ ]:
assert len(patient_ids) == 4
assert identity_summary["integer_rebinding_created_another_object"]
assert identity_summary["list_mutation_kept_identity"]
assert nested_tuple[1] == [78, 82, 86]
assert patient_record["P101"]["values"] == [78, 82, 86]
assert deep_record["P101"]["values"] == [78, 82]
assert unhashable_error == "TypeError"
assert caller_values == [78.0, 82.0, 86.0]
assert unsafe_first is unsafe_second
assert safe_first is not safe_second
assert protocol_summary["length"] == 3
assert measurement_patient == "P101"
assert container_summary["missing"] == ["FiO2"]
assert list(observed_by_patient) == ["P101", "P103", "P104"]
assert value_summary["mean"] == 82.0
print("Python step checks passed.")


## 4. Checks

Python 기본 객체, Data Model, container, 반복, 함수와 예외 처리 결과를 한 번 더 확인합니다. 원본이 예상과 다르게 바뀌었다면 name이 같은 객체에 bind되어 있는지, copy가 중첩 객체까지 분리했는지부터 확인하세요.

In [ ]:
assert sys.version_info[:2] == (3, 11)
assert missing_value is None
assert alias_values is original_values
assert copied_values is not original_values
assert list_identity_before == list_identity_after
assert string_identity_before != string_identity_after
assert list_identity_before_append == list_identity_after_append
assert shallow_record["P101"] is patient_record["P101"]
assert deep_record["P101"] is not patient_record["P101"]
assert measurements_by_key[("P101", "HR")] == 84.5
assert unsafe_collect.__defaults__[0] == [78.0, 82.0]
assert protocol_summary["iterated_items"] == protocol_values
assert expected_parameters - observed_parameters == {"FiO2"}
assert first_value == first
assert last_value == last
assert has_required_columns
assert parsing_error == "ValueError"
print("All appendix checks passed.")


## 5. Next Steps

- 대입과 함수 호출은 객체를 자동으로 복사하지 않습니다. mutable 객체를 다른 name이나 함수에 전달할 때 변경이 어디까지 전파되는지 확인합니다.
- immutable 객체의 재대입과 mutable 객체의 in-place 변경을 `id()`로 구분해 봅니다.
- 중첩된 container는 shallow copy가 안쪽 객체를 공유합니다. 완전히 독립된 복사본이 필요한 경우에만 `copy.deepcopy()`를 사용합니다.
- dict key와 set 원소를 정할 때는 값이 hashable한지 확인하고, 함수 기본값에는 mutable 객체를 직접 두지 않습니다.
- dict와 set은 key 기반 데이터와 coverage 비교에, list와 tuple은 순서가 있는 값에 사용합니다.
- comprehension은 짧은 변환에만 사용하고 복잡한 분기는 이름 있는 함수로 분리합니다.
- 오류가 발생하면 마지막 예외 타입과 traceback의 가장 가까운 사용자 코드 위치부터 읽습니다.
- 다음 notebook에서는 Python container를 label이 있는 Series와 DataFrame으로 확장하고, Index와 GroupBy API를 학습합니다.

## 6. References

이 notebook의 설명과 예제는 다음 공식 문서를 기준으로 작성했습니다.

- [An Informal Introduction to Python](https://docs.python.org/3.11/tutorial/introduction.html)
- [Data model](https://docs.python.org/3.11/reference/datamodel.html)
- [copy — Shallow and deep copy operations](https://docs.python.org/3.11/library/copy.html)
- [Glossary: hashable](https://docs.python.org/3.11/glossary.html#term-hashable)
- [More Control Flow Tools](https://docs.python.org/3.11/tutorial/controlflow.html)
- [Data Structures](https://docs.python.org/3.11/tutorial/datastructures.html)
- [Errors and Exceptions](https://docs.python.org/3.11/tutorial/errors.html)
- [Built-in Functions](https://docs.python.org/3.11/library/functions.html)